# ĐỒ ÁN MÔN HỌC: XỬ LÝ ẢNH — NHẬN DIỆN CHẤT LƯỢNG GẠO
## Phân hệ 4: Phân loại chất lượng & Thống kê số liệu — Thắng SimpLord

Notebook này phân tích phạm vi đặc trưng từ dataset mẫu để thiết lập ngưỡng tham chiếu, phân loại chất lượng và màu sắc cho từng hạt từ đặc trưng của TV3, đồng thời tổng hợp thống kê và trực quan hóa kết quả[cite: 4, 5, 7].

**Đặc tả kỹ thuật:**
* **Quy tắc phân loại (`if - else`):** Phân tích phạm vi giá trị của từng nhóm chất lượng (`Tốt`, `Vỡ`, `Kém`, `Tệ`, `Tạp chất`), xác định ngưỡng kích thước/hình dạng và ngưỡng màu (`Bình thường`, `Vàng`, `Sẫm`), phân loại bằng `if - else` và xử lý trường hợp `"Không xác định"`[cite: 5, 7].
* **Thống kê số lượng & tỷ lệ:** Đếm tổng số hạt, đếm theo nhãn chất lượng và nhãn màu sắc, tính tỷ lệ `(count / total) * 100` cho từng nhóm[cite: 7].
* **Thống kê kích thước, màu sắc & biểu đồ:** Tính kích thước hạt gạo trung bình (`numpy.mean()`), phân tích thống kê `HSV` và vẽ biểu đồ trực quan bằng `Matplotlib`[cite: 7].

In [ ]:
# %% [code]
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN TƯƠNG ĐỐI (ĐỌC TRỰC TIẾP ĐẦU RA TỪ TV3)
# ==============================================================================
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PROJECT_ROOT = os.getcwd()

FEAT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "features_dataset")
REF_FEAT_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "reference_grain_features.csv")
BATCH_FEAT_CLEAN_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "grain_features_clean.csv")
BATCH_FEAT_ALL_CSV_PATH = os.path.join(FEAT_OUTPUT_DIR, "grain_features.csv")

CLS_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "classification_results")
CLASSIFIED_GRAINS_CSV = os.path.join(CLS_OUTPUT_DIR, "classified_grains.csv")
SUMMARY_STATS_CSV = os.path.join(CLS_OUTPUT_DIR, "quality_color_summary.csv")
REF_RANGES_CSV = os.path.join(CLS_OUTPUT_DIR, "reference_quality_ranges.csv")

print(f"Thư mục gốc: {PROJECT_ROOT}")
print(f"Đặc trưng mẫu từ TV3: {REF_FEAT_CSV_PATH}")
print(f"Đặc trưng hạt từ TV3: {BATCH_FEAT_CLEAN_CSV_PATH}")

In [ ]:
# %% [code]
def load_tv3_reference_features(ref_csv_path: str = REF_FEAT_CSV_PATH) -> pd.DataFrame:
    """
    Nạp bảng đặc trưng ảnh gạo đơn lẻ mẫu đã được TV3 trích xuất.
    """
    if not os.path.exists(ref_csv_path):
        raise FileNotFoundError(
            f"Không tìm thấy '{ref_csv_path}'. Hãy chạy phần trích xuất đặc trưng mẫu của TV3 trước."
        )

    df_ref = pd.read_csv(ref_csv_path)
    if "feature_status" in df_ref.columns:
        df_ref = df_ref[df_ref["feature_status"] == "ok"].copy()

    if "label_name" not in df_ref.columns or df_ref.empty:
        raise ValueError("Bảng đặc trưng mẫu của TV3 thiếu cột 'label_name' hoặc không có dữ liệu hợp lệ.")

    return df_ref.reset_index(drop=True)


def analyze_reference_ranges_from_tv3(df_ref: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """
    Phân tích phạm vi giá trị (min, max, mean, median, q25, q75, iqr) của từng nhóm chất lượng
    dựa trên đặc trưng mẫu của TV3, sau đó xây dựng bộ ngưỡng & hồ sơ so sánh.
    """
    metric_cols = [
        "shape_area",
        "shape_length",
        "shape_width",
        "shape_aspect_ratio",
        "shape_circularity",
        "shape_solidity",
        "color_h_mean",
        "color_s_mean",
        "color_v_mean",
        "color_yellow_ratio",
        "color_dark_ratio",
    ]
    available_cols = [c for c in metric_cols if c in df_ref.columns]

    good_df = df_ref[df_ref["label_name"] == "Tốt"]
    ref_base_area = (
        float(good_df["shape_area"].median())
        if (not good_df.empty and "shape_area" in good_df.columns)
        else float(df_ref["shape_area"].median())
    )
    ref_base_length = (
        float(good_df["shape_length"].median())
        if (not good_df.empty and "shape_length" in good_df.columns)
        else float(df_ref["shape_length"].median())
    )

    work_ref = df_ref.copy()
    work_ref["rel_area"] = work_ref["shape_area"] / max(1e-6, ref_base_area)
    work_ref["rel_length"] = work_ref["shape_length"] / max(1e-6, ref_base_length)
    profile_cols = available_cols + ["rel_area", "rel_length"]

    range_rows = []
    class_profiles = {}

    for lbl, grp in work_ref.groupby("label_name"):
        stat_row = {"label_name": lbl, "sample_count": int(len(grp))}
        prof = {}
        for col in profile_cols:
            s = grp[col].dropna()
            if s.empty:
                continue
            stat_row[f"{col}_min"] = round(float(s.min()), 4)
            stat_row[f"{col}_max"] = round(float(s.max()), 4)
            stat_row[f"{col}_mean"] = round(float(s.mean()), 4)
            stat_row[f"{col}_median"] = round(float(s.median()), 4)

            q25 = float(s.quantile(0.25))
            q75 = float(s.quantile(0.75))
            med = float(s.median())
            iqr = max(1e-4, q75 - q25)
            prof[col] = {"min": float(s.min()), "max": float(s.max()), "median": med, "iqr": iqr}

        range_rows.append(stat_row)
        class_profiles[lbl] = prof

    df_ranges = pd.DataFrame(range_rows)

    # Xác định ngưỡng màu (Bình thường, Vàng, Sẫm) từ phạm vi nhóm Tốt vs nhóm lỗi
    if not good_df.empty:
        good_y_q75 = float(good_df["color_yellow_ratio"].quantile(0.75))
        good_d_q75 = float(good_df["color_dark_ratio"].quantile(0.75))
        good_v_q25 = float(good_df["color_v_mean"].quantile(0.25))
        good_sol_q25 = float(good_df["shape_solidity"].quantile(0.25))
    else:
        good_y_q75, good_d_q75, good_v_q25, good_sol_q25 = 0.14, 0.14, 105.0, 0.82

    defect_df = df_ref[df_ref["label_name"].isin(["Kém", "Tệ", "Tạp chất"])]
    if not defect_df.empty:
        def_y_med = float(defect_df["color_yellow_ratio"].median())
        def_d_med = float(defect_df["color_dark_ratio"].median())
        yellow_thresh = float(np.clip((good_y_q75 + def_y_med) / 2.0, good_y_q75 + 0.02, 0.65))
        dark_thresh = float(np.clip((good_d_q75 + def_d_med) / 2.0, good_d_q75 + 0.02, 0.65))
    else:
        yellow_thresh = min(0.55, good_y_q75 + 0.08)
        dark_thresh = min(0.55, good_d_q75 + 0.08)

    # Xác định ngưỡng kích thước tương đối cho hạt Vỡ từ nhóm mẫu Vỡ
    broken_df = work_ref[work_ref["label_name"] == "Vỡ"]
    if not broken_df.empty:
        broken_rel_area_thresh = float(
            np.clip((broken_df["rel_area"].quantile(0.75) + 1.0) / 2.0, 0.45, 0.85)
        )
        broken_rel_len_thresh = float(
            np.clip((broken_df["rel_length"].quantile(0.75) + 1.0) / 2.0, 0.50, 0.88)
        )
    else:
        broken_rel_area_thresh = 0.68
        broken_rel_len_thresh = 0.74

    thresholds = {
        "yellow_ratio_thresh": round(yellow_thresh, 4),
        "dark_ratio_thresh": round(dark_thresh, 4),
        "dark_v_mean_thresh": round(max(60.0, good_v_q25 * 0.82), 2),
        "good_solidity_min": round(max(0.72, min(0.95, good_sol_q25)), 4),
        "broken_rel_area_thresh": round(broken_rel_area_thresh, 4),
        "broken_rel_len_thresh": round(broken_rel_len_thresh, 4),
        "class_profiles": class_profiles,
    }

    return thresholds, df_ranges


df_ref_tv3 = load_tv3_reference_features()
REF_THRESHOLDS, df_ref_ranges = analyze_reference_ranges_from_tv3(df_ref_tv3)

print("Các ngưỡng suy ra từ đặc trưng mẫu của TV3:")
for k, v in REF_THRESHOLDS.items():
    if k != "class_profiles":
        print(f"  - {k}: {v}")
df_ref_ranges[["label_name", "sample_count", "shape_area_mean", "shape_length_mean", "color_yellow_ratio_mean", "color_dark_ratio_mean"]]

In [8]:
# %% [code]
def classify_color_label(row: pd.Series, thresholds: dict) -> str:
    """
    Phân loại nhãn màu ('Bình thường', 'Vàng', 'Sẫm', 'Không xác định') bằng if - else.
    """
    h_val = row.get("color_h_mean", np.nan)
    s_val = row.get("color_s_mean", np.nan)
    v_val = row.get("color_v_mean", np.nan)
    y_ratio = row.get("color_yellow_ratio", np.nan)
    d_ratio = row.get("color_dark_ratio", np.nan)

    if any(pd.isna(x) or np.isinf(x) for x in [h_val, s_val, v_val, y_ratio, d_ratio]):
        return "Không xác định"

    y_th = float(thresholds["yellow_ratio_thresh"])
    d_th = float(thresholds["dark_ratio_thresh"])
    v_th = float(thresholds["dark_v_mean_thresh"])

    is_yellow = (y_ratio >= y_th) or ((15.0 <= h_val <= 38.0) and (s_val >= 50.0) and (y_ratio >= y_th * 0.75))
    is_dark = (d_ratio >= d_th) or ((v_val <= v_th) and (d_ratio >= d_th * 0.65))

    if is_yellow and is_dark:
        if y_ratio >= d_ratio:
            return "Vàng"
        else:
            return "Sẫm"
    elif is_yellow:
        return "Vàng"
    elif is_dark:
        return "Sẫm"
    elif 0.0 <= v_val <= 255.0 and 0.0 <= s_val <= 255.0:
        return "Bình thường"
    else:
        return "Không xác định"


def _distance_to_class_profile(sample_dict: dict, profile: dict) -> float:
    """
    Tính độ lệch chuẩn hóa giữa đặc trưng của hạt với hồ sơ một nhóm mẫu từ TV3.
    """
    weights = {
        "rel_area": 1.6,
        "rel_length": 1.6,
        "shape_aspect_ratio": 1.2,
        "shape_solidity": 1.2,
        "color_h_mean": 1.0,
        "color_s_mean": 1.2,
        "color_v_mean": 1.3,
        "color_yellow_ratio": 1.8,
        "color_dark_ratio": 1.8,
    }
    dist_sum = 0.0
    w_sum = 0.0

    for col, w in weights.items():
        if col not in sample_dict or col not in profile:
            continue
        val = sample_dict[col]
        if pd.isna(val) or np.isinf(val):
            continue
        med = profile[col]["median"]
        scale = max(profile[col]["iqr"], abs(med) * 0.15, 0.05)
        dist_sum += w * abs(val - med) / scale
        w_sum += w

    return (dist_sum / w_sum) if w_sum > 0 else float("inf")


def classify_quality_label(
    row: pd.Series,
    batch_base_area: float,
    batch_base_length: float,
    thresholds: dict,
) -> str:
    """
    Phân loại chất lượng hạt ('Tốt', 'Vỡ', 'Kém', 'Tệ', 'Tạp chất', 'Không xác định')
    bằng cấu trúc if - else đối chiếu đặc trưng TV3 với phạm vi mẫu.
    """
    area = row.get("shape_area", np.nan)
    length = row.get("shape_length", np.nan)
    aspect = row.get("shape_aspect_ratio", np.nan)
    solidity = row.get("shape_solidity", np.nan)
    h_val = row.get("color_h_mean", np.nan)
    s_val = row.get("color_s_mean", np.nan)
    v_val = row.get("color_v_mean", np.nan)
    y_ratio = row.get("color_yellow_ratio", np.nan)
    d_ratio = row.get("color_dark_ratio", np.nan)
    color_lbl = row.get("color_label", "Không xác định")

    # 1. Xử lý trường hợp thiếu / bất thường / không xác định
    if any(pd.isna(x) or np.isinf(x) for x in [area, length, aspect, solidity, v_val, y_ratio, d_ratio]):
        return "Không xác định"
    if area <= 0 or length <= 0 or color_lbl == "Không xác định":
        return "Không xác định"

    rel_area = float(area) / max(1e-6, batch_base_area)
    rel_length = float(length) / max(1e-6, batch_base_length)
    profiles = thresholds.get("class_profiles", {})
    broken_area_th = float(thresholds["broken_rel_area_thresh"])
    broken_len_th = float(thresholds["broken_rel_len_thresh"])
    good_sol_min = float(thresholds["good_solidity_min"])

    sample_vec_dict = {
        "rel_area": rel_area,
        "rel_length": rel_length,
        "shape_aspect_ratio": float(aspect),
        "shape_solidity": float(solidity),
        "color_h_mean": float(h_val),
        "color_s_mean": float(s_val),
        "color_v_mean": float(v_val),
        "color_yellow_ratio": float(y_ratio),
        "color_dark_ratio": float(d_ratio),
    }

    # 2. Kiểm tra điều kiện bằng if - else đối chiếu ngưỡng & hồ sơ mẫu
    if (aspect < 1.25 and rel_area > 1.75) or (solidity < 0.62):
        if "Tạp chất" in profiles:
            return "Tạp chất"
        else:
            return "Tệ"
    elif (rel_area < broken_area_th) and (rel_length < broken_len_th):
        return "Vỡ"
    elif (color_lbl == "Bình thường") and (rel_area >= broken_area_th) and (solidity >= good_sol_min):
        if "Tốt" in profiles:
            d_good = _distance_to_class_profile(sample_vec_dict, profiles["Tốt"])
            if d_good <= 2.5:
                return "Tốt"
            else:
                return "Kém"
        else:
            return "Tốt"
    else:
        if not profiles:
            return "Không xác định"

        best_lbl = "Không xác định"
        best_dist = float("inf")
        for lbl, prof in profiles.items():
            d = _distance_to_class_profile(sample_vec_dict, prof)
            if d < best_dist:
                best_dist = d
                best_lbl = lbl

        if best_dist > 4.0:
            return "Không xác định"
        elif best_lbl in ("Tốt", "Vỡ", "Kém", "Tệ", "Tạp chất"):
            return best_lbl
        else:
            return "Không xác định"

In [ ]:
# %% [code]
def load_tv3_batch_features() -> pd.DataFrame:
    if os.path.exists(BATCH_FEAT_CLEAN_CSV_PATH):
        return pd.read_csv(BATCH_FEAT_CLEAN_CSV_PATH)
    if os.path.exists(BATCH_FEAT_ALL_CSV_PATH):
        return pd.read_csv(BATCH_FEAT_ALL_CSV_PATH)
    raise FileNotFoundError("Không tìm thấy file đặc trưng hạt của TV3.")


def run_tv4_classification_and_stats(thresholds: dict = REF_THRESHOLDS) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    df = load_tv3_batch_features().copy()
    os.makedirs(CLS_OUTPUT_DIR, exist_ok=True)

    valid_area = df["shape_area"].dropna()
    valid_len = df["shape_length"].dropna()
    batch_base_area = float(valid_area.quantile(0.65)) if len(valid_area) > 0 else 1.0
    batch_base_length = float(valid_len.quantile(0.65)) if len(valid_len) > 0 else 1.0

    df["color_label"] = df.apply(lambda r: classify_color_label(r, thresholds), axis=1)
    df["quality_label"] = df.apply(
        lambda r: classify_quality_label(r, batch_base_area, batch_base_length, thresholds),
        axis=1,
    )

    total_grains = len(df)
    avg_area = float(np.mean(df["shape_area"])) if total_grains > 0 else 0.0
    avg_length = float(np.mean(df["shape_length"])) if total_grains > 0 else 0.0
    avg_width = float(np.mean(df["shape_width"])) if total_grains > 0 else 0.0

    quality_order = ["Tốt", "Vỡ", "Kém", "Tệ", "Tạp chất", "Không xác định"]
    color_order = ["Bình thường", "Vàng", "Sẫm", "Không xác định"]

    q_counts = df["quality_label"].value_counts().to_dict()
    c_counts = df["color_label"].value_counts().to_dict()

    summary_rows = []
    for lbl in quality_order:
        cnt = int(q_counts.get(lbl, 0))
        pct = float((cnt / total_grains) * 100.0) if total_grains > 0 else 0.0
        sub = df[df["quality_label"] == lbl]
        summary_rows.append({
            "group_type": "Chất lượng",
            "label": lbl,
            "count": cnt,
            "percentage": round(pct, 2),
            "mean_area_px": round(float(np.mean(sub["shape_area"])), 2) if len(sub) > 0 else np.nan,
            "mean_length_px": round(float(np.mean(sub["shape_length"])), 2) if len(sub) > 0 else np.nan,
        })

    for lbl in color_order:
        cnt = int(c_counts.get(lbl, 0))
        pct = float((cnt / total_grains) * 100.0) if total_grains > 0 else 0.0
        sub = df[df["color_label"] == lbl]
        summary_rows.append({
            "group_type": "Màu sắc",
            "label": lbl,
            "count": cnt,
            "percentage": round(pct, 2),
            "mean_area_px": round(float(np.mean(sub["shape_area"])), 2) if len(sub) > 0 else np.nan,
            "mean_length_px": round(float(np.mean(sub["shape_length"])), 2) if len(sub) > 0 else np.nan,
        })

    df_summary = pd.DataFrame(summary_rows)

    hsv_stats = {
        "H_mean": round(float(np.mean(df["color_h_mean"])), 2) if total_grains > 0 else np.nan,
        "H_std": round(float(np.std(df["color_h_mean"])), 2) if total_grains > 0 else np.nan,
        "S_mean": round(float(np.mean(df["color_s_mean"])), 2) if total_grains > 0 else np.nan,
        "S_std": round(float(np.std(df["color_s_mean"])), 2) if total_grains > 0 else np.nan,
        "V_mean": round(float(np.mean(df["color_v_mean"])), 2) if total_grains > 0 else np.nan,
        "V_std": round(float(np.std(df["color_v_mean"])), 2) if total_grains > 0 else np.nan,
    }

    report = {
        "total_grains": int(total_grains),
        "avg_area_px": round(avg_area, 2),
        "avg_length_px": round(avg_length, 2),
        "avg_width_px": round(avg_width, 2),
        "quality_counts": {k: int(q_counts.get(k, 0)) for k in quality_order},
        "color_counts": {k: int(c_counts.get(k, 0)) for k in color_order},
        "hsv_stats": hsv_stats,
    }

    df.to_csv(CLASSIFIED_GRAINS_CSV, index=False, encoding="utf-8-sig")
    df_summary.to_csv(SUMMARY_STATS_CSV, index=False, encoding="utf-8-sig")
    df_ref_ranges.to_csv(REF_RANGES_CSV, index=False, encoding="utf-8-sig")

    print("=" * 65)
    print("THỐNG KÊ CHẤT LƯỢNG & MÀU SẮC")
    print("=" * 65)
    print(f"Tổng số hạt              : {report['total_grains']}")
    print(f"Diện tích trung bình (px): {report['avg_area_px']}")
    print(f"Chiều dài trung bình (px): {report['avg_length_px']}")
    print(f"Chiều rộng trung bình(px): {report['avg_width_px']}")
    print("-" * 65)
    print("Phân bố chất lượng:")
    for lbl in quality_order:
        c = report["quality_counts"][lbl]
        p = (c / total_grains * 100.0) if total_grains > 0 else 0.0
        print(f"  - {lbl:<15}: {c:>5} hạt ({p:6.2f}%)")
    print("-" * 65)
    print("Phân bố màu sắc:")
    for lbl in color_order:
        c = report["color_counts"][lbl]
        p = (c / total_grains * 100.0) if total_grains > 0 else 0.0
        print(f"  - {lbl:<15}: {c:>5} hạt ({p:6.2f}%)")
    print("-" * 65)
    print("Thống kê HSV:", hsv_stats)
    print("=" * 65)

    return df, df_summary, report


df_classified, df_summary_stats, stats_report = run_tv4_classification_and_stats()
df_summary_stats

In [ ]:
# %% [code]
def plot_tv4_charts(df: pd.DataFrame, report: dict) -> None:
    if df.empty:
        print("Không có dữ liệu để vẽ biểu đồ.")
        return

    quality_labels = ["Tốt", "Vỡ", "Kém", "Tệ", "Tạp chất", "Không xác định"]
    q_counts = [report["quality_counts"].get(k, 0) for k in quality_labels]

    color_labels = ["Bình thường", "Vàng", "Sẫm", "Không xác định"]
    c_counts = [report["color_counts"].get(k, 0) for k in color_labels]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].bar(quality_labels, q_counts, color=["#2E8B57", "#E69F00", "#56B4E9", "#D55E00", "#7F7F7F", "#BDBDBD"])
    axes[0, 0].set_title("Số lượng hạt theo nhóm chất lượng")
    axes[0, 0].set_ylabel("Số lượng hạt")
    axes[0, 0].tick_params(axis="x", rotation=20)
    for idx, val in enumerate(q_counts):
        axes[0, 0].text(idx, val + max(1, max(q_counts) * 0.01), str(val), ha="center", va="bottom", fontsize=9)

    axes[0, 1].bar(color_labels, c_counts, color=["#4CAF50", "#FBC02D", "#5D4037", "#9E9E9E"])
    axes[0, 1].set_title("Số lượng hạt theo nhãn màu sắc")
    axes[0, 1].set_ylabel("Số lượng hạt")
    axes[0, 1].tick_params(axis="x", rotation=15)
    for idx, val in enumerate(c_counts):
        axes[0, 1].text(idx, val + max(1, max(c_counts) * 0.01), str(val), ha="center", va="bottom", fontsize=9)

    axes[1, 0].hist(df["color_v_mean"].dropna(), bins=25, alpha=0.75, label="Độ sáng (V)", color="#DD8452")
    axes[1, 0].hist(df["color_s_mean"].dropna(), bins=25, alpha=0.65, label="Độ bão hòa (S)", color="#4C72B0")
    axes[1, 0].set_title("Phân bố độ sáng (V) và độ bão hòa (S) trong vùng hạt")
    axes[1, 0].set_xlabel("Giá trị kênh (0 - 255)")
    axes[1, 0].set_ylabel("Số lượng hạt")
    axes[1, 0].legend()

    valid_groups = [g for g in quality_labels if (df["quality_label"] == g).sum() > 0]
    box_data = [df.loc[df["quality_label"] == g, "color_h_mean"].dropna().values for g in valid_groups]
    if box_data:
        axes[1, 1].boxplot(box_data, tick_labels=valid_groups)
        axes[1, 1].set_title("Phân bố sắc độ (H) theo nhóm chất lượng")
        axes[1, 1].set_ylabel("Hue (0 - 179)")
        axes[1, 1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()


plot_tv4_charts(df_classified, stats_report)